### Initial ML Model for Delsys Data Input (LDA Model)
Consists of:
- Filtering
- Windowing
- Feature Extraction
- Training LDA
- Evaluating accuracy
- Works for ADLs like “water-bottle lift” and “zipper”

First setting up virtual environment:
- py -m venv venv
- venv\Scripts\activate

Then setting up dependencies:
- pip install numpy scipy scikit-learn matplotlib seaborn

##### Imports

In [67]:
import pandas as pd
import numpy as np
import re
from scipy.signal import butter, filtfilt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import classification_report, confusion_matrix

##### Functions

In [68]:
def load_emg(csv_path):
    """
    Load the first 4 EMG sensors and their timestamps from a Delsys Trigno CSV.
    Keeps the timestamps exactly as they appear in the file.
    Converts all data to numeric for downstream processing.
    """
    # Load CSV with header row at line 6 (index 5) and skip rows 7 and 8
    df = pd.read_csv(csv_path, header=5, skiprows=[6, 7], engine="python")
    
    # Remove leading/trailing spaces from column names
    df.columns = df.columns.str.strip()
    
    # Select first 4 EMG sensors and their corresponding timestamps
    emg_cols = [col for col in df.columns if "(mV)" in col][:4]          
    time_cols = [col for col in df.columns if "Time Series" in col][:3]  
    
    # Convert to numeric, coerce errors, fill NaNs with 0
    emg_data = df[emg_cols].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy()
    time_data = df[time_cols].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy()
    
    return emg_data, time_data


In [69]:
#def bandpass_filter(data, lowcut=20, highcut=450, fs=963, order=4):
#    """
#    Apply zero-phase 4th-order Butterworth bandpass filter to EMG data.
#    """
#    nyq = 0.5 * fs
#    low = lowcut / nyq
#    high = highcut / nyq
#    b, a = butter(order, [low, high], btype='band')
#    filtered = filtfilt(b, a, data, axis=0)
#    return filtered

In [70]:
def extract_features(emg_window):
    """
    Extract time-domain EMG features for a window:
        - MAV: mean absolute value
        - RMS: root mean square
        - WL: waveform length
    """
    mav = np.mean(np.abs(emg_window), axis=0)
    rms = np.sqrt(np.mean(emg_window**2, axis=0))
    wl = np.sum(np.abs(np.diff(emg_window, axis=0)), axis=0)
    return np.concatenate([mav, rms, wl])

In [71]:
def window_emg(emg_data, fs=963, window_sec=0.2, overlap_sec=0.1):
    """
    Slice EMG data into overlapping windows and extract features.
    Returns:
        features: (n_windows, n_features)
    """
    win_size = int(window_sec * fs)
    step = int((window_sec - overlap_sec) * fs)
    features = []

    for start in range(0, emg_data.shape[0] - win_size, step):
        window = emg_data[start:start + win_size]
        feat = extract_features(window)
        features.append(feat)
    
    return np.array(features)

In [72]:
def process_trial(csv_path, label, fs=963):
    """
    Load EMG, apply filtering, windowing, and return features + labels.
    """
    emg_data, _ = load_emg(csv_path)
    X = window_emg(emg_data, fs=fs)
    y = np.full(X.shape[0], label)
    return X, y

In [73]:
def load_all_trials(trial_files, labels):
    """
    trial_files: list of CSV paths
    labels: list of integer labels for each trial
    """
    X_list, y_list = [], []
    for file, label in zip(trial_files, labels):
        X_trial, y_trial = process_trial(file, label)
        X_list.append(X_trial)
        y_list.append(y_trial)
    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    return X, y

In [74]:
def prepare_data(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test

In [75]:
def train_evaluate(X_train, X_test, y_train, y_test):
    clf = LinearDiscriminantAnalysis()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    return clf

##### Load and process ADL data

X_lift, y_lift = process_trial("20251113-Data/Lifting.csv", label=0)
X_zip, y_zip = process_trial("20251113-Data/Zipping.csv", label=1)
X_pinch, y_pinch = process_trial("20251113-Data/Pinching.csv", label=2)

X = np.vstack([X_lift, X_zip, X_pinch])
y = np.concatenate([y_lift, y_zip, y_pinch])

In [76]:
if __name__ == "__main__":
    trial_files = [
        "20251113-Data/Lifting.csv",
        "20251113-Data/Zipping.csv",
        "20251113-Data/Pinching.csv"
    ]
    labels = [0, 1, 2]

    # Load and process all trials
    X, y = load_all_trials(trial_files, labels)

    # Split and scale data
    X_train, X_test, y_train, y_test = prepare_data(X, y)

    # Train and evaluate
    clf = train_evaluate(X_train, X_test, y_train, y_test)

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.09      0.16       799
           1       0.38      0.98      0.55       803
           2       0.88      0.12      0.20       632

    accuracy                           0.42      2234
   macro avg       0.71      0.40      0.30      2234
weighted avg       0.69      0.42      0.31      2234

Confusion Matrix:
 [[ 69 724   6]
 [  9 790   4]
 [  2 557  73]]
